# Part B: KRW/CHF Exchange Rate Analysis
## South Korea (KRW) ↔ Switzerland (CHF)

This notebook analyzes the exchange rate change between the Korean Won and Swiss Franc over a 6-month period.

**Analysis Period:** August 2025 - January 2026

**Objectives:**
1. Fetch KRW/CHF historical data
2. Calculate percentage change
3. Analyze impact on Korean exporters and importers
4. Visualize trends

## Setup

In [1]:
# Import libraries
import sys
sys.path.append('../src')

from data_fetchers import ForexDataFetcher
from calculators import ExchangeRateCalculator
from visualizers import ForexVisualizer

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Enable inline plotting
%matplotlib inline

print("✅ Libraries imported successfully")

ModuleNotFoundError: No module named 'yfinance'

## Step 1: Fetch KRW/CHF Exchange Rate Data

We'll use **yfinance** to fetch exchange rate data. This is FREE and requires no API key.

**How it works:**
1. Fetch KRW/USD rate
2. Fetch CHF/USD rate
3. Calculate cross rate: 1 KRW = X CHF

In [ ]:
# Configuration
BASE_CURRENCY = 'KRW'      # South Korean Won
TARGET_CURRENCY = 'CHF'    # Swiss Franc
START_DATE = '2025-08-01'
END_DATE = '2026-01-30'

print(f"Fetching {BASE_CURRENCY}/{TARGET_CURRENCY} data from {START_DATE} to {END_DATE}...")

# Fetch data
fetcher = ForexDataFetcher()
forex_data = fetcher.get_exchange_rate_yfinance(
    BASE_CURRENCY, 
    TARGET_CURRENCY, 
    START_DATE, 
    END_DATE
)

print(f"\n✅ Fetched {len(forex_data)} days of data")
print(f"   Date range: {forex_data.index[0]} to {forex_data.index[-1]}")

## Step 2: Explore the Data

Let's look at the structure and preview the data.

In [ ]:
# Display first few rows
print("First 5 days:")
print(forex_data.head())

print("\nLast 5 days:")
print(forex_data.tail())

In [ ]:
# Statistical summary
print("Statistical Summary of KRW/CHF Cross Rate:")
print(forex_data['cross_rate'].describe())

## Step 3: Calculate Exchange Rate Change

**Formula:**
```
Percentage Change = ((New Rate - Old Rate) / Old Rate) × 100
```

**Expected Result (from project document):**
- Historical: 1 KRW = 0.00067 CHF
- Current: 1 KRW = 0.00062 CHF
- Change: -7.46% (KRW depreciation)

In [ ]:
# Get historical and current rates
historical_rate = forex_data['cross_rate'].iloc[0]
current_rate = forex_data['cross_rate'].iloc[-1]

# Calculate change using our calculator
calculator = ExchangeRateCalculator()
change_metrics = calculator.calculate_percentage_change(historical_rate, current_rate)

# Display results
print("="*70)
print("EXCHANGE RATE CHANGE ANALYSIS: KRW/CHF")
print("="*70)
print(f"\nHistorical Rate ({forex_data.index[0].strftime('%B %d, %Y')}):")
print(f"  1 KRW = {change_metrics['old_rate']:.6f} CHF")

print(f"\nCurrent Rate ({forex_data.index[-1].strftime('%B %d, %Y')}):")
print(f"  1 KRW = {change_metrics['new_rate']:.6f} CHF")

print(f"\nChange:")
print(f"  Absolute: {change_metrics['absolute_change']:.6f} CHF")
print(f"  Percentage: {change_metrics['percentage_change']:.2f}%")
print(f"  Direction: {change_metrics['direction'].upper()}")

print("\n" + "="*70)

## Step 4: Interpret the Results

### What does KRW depreciation mean?

If the Korean Won has **depreciated** (negative % change):
- **1 KRW buys fewer CHF** than before
- Korean goods become **cheaper** for Swiss buyers
- Swiss goods become **more expensive** for Korean buyers

In [ ]:
# Interpretation
if change_metrics['percentage_change'] < 0:
    print("🔴 The Korean Won has DEPRECIATED against the Swiss Franc")
    print(f"\nWhat this means:")
    print(f"  • Swiss products (watches, pharmaceuticals) are ~{abs(change_metrics['percentage_change']):.2f}% MORE expensive for Koreans")
    print(f"  • Korean exports (Samsung electronics, Hyundai cars) are ~{abs(change_metrics['percentage_change']):.2f}% CHEAPER for Swiss buyers")
    print(f"\nExample:")
    print(f"  • A Swiss watch that cost 1M CHF = {1000000/historical_rate:,.0f} KRW before")
    print(f"  • Now costs {1000000/current_rate:,.0f} KRW ({((1000000/current_rate)-(1000000/historical_rate))/(1000000/historical_rate)*100:.1f}% increase)")
else:
    print("🟢 The Korean Won has APPRECIATED against the Swiss Franc")
    print(f"\nWhat this means:")
    print(f"  • Swiss products are ~{change_metrics['percentage_change']:.2f}% CHEAPER for Koreans")
    print(f"  • Korean exports are ~{change_metrics['percentage_change']:.2f}% MORE expensive for foreign buyers")

## Step 5: Stakeholder Impact Analysis

### Who wins and who loses from KRW depreciation?

In [ ]:
# Analyze impact on stakeholders
impact = calculator.analyze_impact(change_metrics['percentage_change'], BASE_CURRENCY, TARGET_CURRENCY)

print("="*70)
print("STAKEHOLDER IMPACT ANALYSIS")
print("="*70)

print("\n🟢 KOREAN EXPORTERS (Samsung, Hyundai, LG)")
print(f"   Impact: {impact['exporters']['impact']}")
print(f"   Magnitude: {impact['exporters']['magnitude']:.2f}%")
print(f"   ➜ {impact['exporters']['reason']}")
if 'examples' in impact['exporters']:
    print(f"   ➜ Example: {impact['exporters']['examples']['korea']}")

print("\n🔴 KOREAN IMPORTERS (Pharmaceutical Distributors, Machinery Buyers)")
print(f"   Impact: {impact['importers']['impact']}")
print(f"   Magnitude: {impact['importers']['magnitude']:.2f}%")
print(f"   ➜ {impact['importers']['reason']}")
if 'examples' in impact['importers']:
    print(f"   ➜ Example: {impact['importers']['examples']['korea']}")

print("\n🟡 KOREAN CHAEBOLS (Conglomerates like Samsung)")
print(f"   Impact: MIXED")
print(f"   ➜ Export divisions (semiconductors) BENEFIT from cheaper products")
print(f"   ➜ Import operations (Swiss machinery) face HIGHER costs")
print(f"   ➜ Net effect depends on export vs import intensity")

print("\n" + "="*70)

## Step 6: Visualize Exchange Rate Trend

Let's create a professional time series chart showing the KRW/CHF rate over 6 months.

In [ ]:
# Create visualizer
visualizer = ForexVisualizer()

# Plot time series
visualizer.plot_exchange_rate_time_series(
    forex_data,
    BASE_CURRENCY,
    TARGET_CURRENCY,
    save_path='../outputs/charts/notebook_exchange_rate.png'
)

## Step 7: Volatility Analysis

How volatile has the KRW/CHF rate been?

In [ ]:
# Calculate daily returns
returns = forex_data['cross_rate'].pct_change().dropna()

# Statistics
print("Daily Return Statistics:")
print(f"  Mean return: {returns.mean()*100:.3f}%")
print(f"  Std deviation: {returns.std()*100:.3f}%")
print(f"  Min (worst day): {returns.min()*100:.3f}%")
print(f"  Max (best day): {returns.max()*100:.3f}%")

# Annualized volatility
annualized_vol = returns.std() * np.sqrt(252)  # 252 trading days
print(f"\nAnnualized volatility: {annualized_vol*100:.2f}%")

In [ ]:
# Visualize volatility
visualizer.plot_volatility_analysis(
    forex_data,
    BASE_CURRENCY,
    TARGET_CURRENCY,
    save_path='../outputs/charts/notebook_volatility.png'
)

## Step 8: Summary and Key Takeaways

Let's create a summary DataFrame of all our findings.

In [ ]:
# Create summary DataFrame
summary = pd.DataFrame({
    'Metric': [
        'Analysis Period',
        'Currency Pair',
        'Historical Rate',
        'Current Rate',
        'Absolute Change',
        'Percentage Change',
        'Direction',
        'Annualized Volatility',
        'Exporter Impact',
        'Importer Impact'
    ],
    'Value': [
        f"{forex_data.index[0].strftime('%Y-%m-%d')} to {forex_data.index[-1].strftime('%Y-%m-%d')}",
        f"{BASE_CURRENCY}/{TARGET_CURRENCY}",
        f"{historical_rate:.6f} CHF",
        f"{current_rate:.6f} CHF",
        f"{change_metrics['absolute_change']:.6f} CHF",
        f"{change_metrics['percentage_change']:.2f}%",
        change_metrics['direction'].title(),
        f"{annualized_vol*100:.2f}%",
        impact['exporters']['impact'],
        impact['importers']['impact']
    ]
})

print("\n" + "="*70)
print("SUMMARY OF FINDINGS")
print("="*70)
print(summary.to_string(index=False))
print("="*70)

## Key Takeaways

Based on our analysis:

### 1. Exchange Rate Movement
- The Korean Won has depreciated against the Swiss Franc
- This reflects broader economic fundamentals (interest rates, inflation)

### 2. Winners: Korean Exporters
- **Samsung Electronics**: Semiconductors become more competitive in Swiss/European markets
- **Hyundai**: Cars cheaper for foreign buyers
- **LG**: Consumer electronics gain price advantage

### 3. Losers: Korean Importers
- **Pharmaceutical distributors**: Roche and Novartis drugs cost more KRW
- **Machinery importers**: Swiss precision equipment becomes expensive
- **Korean manufacturers**: Higher input costs for Swiss components

### 4. Policy Implications
- Bank of Korea faces trade-offs: support exporters vs. control imported inflation
- SMEs need hedging support to manage currency risk
- Long-term: Won depreciation may improve trade balance (if Marshall-Lerner holds)

---

**Next Steps:**
- Move to **Part C** notebook for Interest Rate Parity (IRP) analysis
- Explore why Korea's 3.25% rate vs Switzerland's 1.25% affects the exchange rate

## Optional: Export Results

Save the summary to Excel for further analysis.

In [ ]:
# Export to Excel
try:
    with pd.ExcelWriter('../outputs/excel/part_b_results.xlsx', engine='openpyxl') as writer:
        summary.to_excel(writer, sheet_name='Summary', index=False)
        forex_data.to_excel(writer, sheet_name='Exchange Rates')
        returns.to_frame(name='Daily Returns').to_excel(writer, sheet_name='Returns')
    
    print("✅ Results exported to ../outputs/excel/part_b_results.xlsx")
except Exception as e:
    print(f"⚠️ Excel export failed: {e}")